# Урок 13. Random Forest и ансамбли
**Библиотеки:** numpy, sklearn, matplotlib

**Цель:** понять идею «толпа умнее одиночки» и feature importance.

## Простыми словами
Одно дерево нестабильно и зубрит. Идея ансамбля: спросить у СОТНИ разных деревьев
и усреднить ответы. Ошибки отдельных деревьев взаимно гасятся.

Чтобы деревья были РАЗНЫМИ, Random Forest:
1. Каждому дереву даёт случайную подвыборку данных (bootstrap).
2. В каждом узле разрешает смотреть только на случайную часть признаков.

**Gradient Boosting (XGBoost/LightGBM)** — другая идея: деревья строятся ПО ОЧЕРЕДИ,
каждое следующее исправляет ошибки предыдущих. Часто топ-результат на табличных данных.

**Feature importance** — лес умеет говорить, какие признаки реально влияли на ответ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(8)
n = 400
sleep  = rng.uniform(3, 9, n)
study  = rng.uniform(0, 10, n)
luck   = rng.uniform(0, 1, n)              # бесполезный признак-обманка
passed = (((study > 4) & (sleep > 5)) | (study > 8)).astype(int)
passed ^= (rng.uniform(0, 1, n) < 0.08).astype(int)

X = np.column_stack([sleep, study, luck])
X_tr, X_te, y_tr, y_te = train_test_split(X, passed, test_size=0.3, random_state=42)

tree   = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr, y_tr)

print(f"Одно дерево : train={tree.score(X_tr,y_tr):.2f}  test={tree.score(X_te,y_te):.2f}")
print(f"Лес (200)   : train={forest.score(X_tr,y_tr):.2f}  test={forest.score(X_te,y_te):.2f}")

In [ ]:
# Feature importance: что реально влияет
names = ['сон', 'учёба', 'удача(шум)']
plt.barh(names, forest.feature_importances_)
plt.title('Важность признаков по мнению леса'); plt.grid(True, axis='x'); plt.show()
for n_, v in zip(names, forest.feature_importances_):
    print(f"{n_}: {v:.2f}")

## Что произошло
Лес обогнал одиночное дерево на test — усреднение 200 мнений устойчивее к шуму.
Feature importance: «учёба» главная, «сон» второй, «удача» почти 0 — лес распознал признак-пустышку.
В реальных проектах этот график — первое, что показывают заказчику.

## Типичные ошибки
1. Слепо верить importance при сильно скоррелированных признаках — важность «размазывается» между ними.
2. Ставить n_estimators=5 — мало деревьев, нет эффекта толпы. 100–300 — нормальный старт.
3. Забывать, что лес тяжелее и медленнее одного дерева — за качество платим ресурсами.

## Конспект 📓
RF = много разных деревьев + голосование. Разнообразие: случайные данные + случайные признаки.
Меньше overfit, чем одно дерево. feature_importances_ — важность признаков.
Boosting (XGBoost) — деревья по очереди исправляют ошибки друг друга.

---
## Мини-задание 💪
Построй график: test accuracy от n_estimators (5, 20, 50, 100, 300). Где насыщение?

## 5 проверочных вопросов ❓
1. Почему лес работает лучше одного дерева?
2. Откуда берётся «разность» деревьев в лесу?
3. Чем boosting отличается от random forest по идее?
4. Что показывает feature_importances_?
5. Лес или одиночное дерево больше склонны к переобучению?

## Домашнее задание 🏠
Задачи 35, 46 из practice_tasks.md.
